In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model = "gpt-5-mini")

#### hard-coded value

In [2]:
messages = [
    {"role": "system", "content": "You are a Genz Poet."},
    {"role": "user", "content": "Write a poem about the beauty of nature."}
]
response = llm.invoke(messages)

response.content

'morning is unedited—light spills like a secret,\ngold seeping under the door the way hope does\nafter a long night.\n\ndew sits like tiny bookmarks on the grass,\neach blade holding a world in reflection.\na robin drops a chorus into the quiet\nand the whole yard leans in to listen.\n\nmountains are old playlists shuffled into place,\neach ridge a remembered refrain; clouds remix them,\nsoft and relentless. the river scrolls and refreshes,\nhanding back the day in glass and motion.\n\nbark is a braille language for hands that want to read,\npine resin smells like memory; salt in the air tastes\nlike stories we have yet to tell. wind passes by\nand everyone hears the same joke: we are here.\n\nwhen night pins up its velvet—stars like unread notifications—\nthe moon swipes through, gentle, inevitable.\nno filter, no captions, no need for likes:\nnature is the OG aesthetic, raw and infinite.\n\nstep outside—plug into the old network:\nit has no buffering and it always logs you in.'

#### Dynamic Template for different types of Messages like Human Message, AI Message, Tool Message, etc.

In [ ]:
my_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a Genz Poet."),
        ("user", "Write a poem about the {topic}.")
    ]
)

nature_prompt = my_template.invoke({"topic": "beauty of nature"})

ChatPromptValue(messages=[SystemMessage(content='You are a Genz Poet.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Write a poem about the beauty of nature.', additional_kwargs={}, response_metadata={})])

### Chains

In [10]:
# Step 1: Create a prompt template
my_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a Genz Poet."),
        ("user", "Write a poem about the {topic}.")
    ]
)


# Step 2: Sending the prompt to the LLM
from pydantic import BaseModel, Field

class llm_schema(BaseModel):
    query: str = Field(description = "The query that was asked to the LLM")
    answer: str = Field(description = "The answer generated by the LLM")
    total_tokens: int  = Field(description = "The total number of tokens used in the LLM response.")
    
llm_structured = llm.with_structured_output(llm_schema)


# Step 3: Parsing the Output
from langchain_core.output_parsers import PydanticOutputParser
parser = PydanticOutputParser(pydantic_object = llm_schema)


# Step 4: Putting it all together
my_chain = my_template | llm_structured 


In [11]:
my_chain.invoke({"topic": "beauty of nature"})

llm_schema(query='Write a poem about the beauty of nature.', answer='morning is a soft alarm\ngreen sneaks between concrete cracks\na river scrolls secrets in light\nbirds draft emails in the sky\nmy chest learns to hold oxygen like prayer\ndawn is a playlist on repeat\ndew is jeweled confession on grass\nwe are small and wide-eyed, still capable\nof stopping — truly stopping —\nand listening: wind translates what the city forgets.\nthis is beauty: patient, loud, patient again.\nwe remember to be alive.', total_tokens=97)